# IOS Risk — Project 03 Evaluation

Compares the untouched Llama base model with the v3 adapter on the same 276
leakage-controlled cases: 50 counterfactual risk assessments, 200 held-out
transaction classifications, and 26 held-out regulatory sections.

**Required inputs**

- `ios-risk-eval-assets-v3`: `testset.json` and `domain_eval.py`
- the completed notebook output containing `Llama-3.1-8B-IOS-Risk-v1`

Use a T4 GPU. The notebook stops immediately if either exact input is absent.


### Step 1: GPU check — fail in seconds, not hours

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {name}  sm_{major}{minor}  ({torch.cuda.device_count()} visible)")

# Kaggle's API cannot set the accelerator, so an API-pushed notebook lands on a
# P100 (sm_60) which Unsloth cannot use. Stop here rather than hang on load.
assert major >= 7, (
    f"{name} is sm_{major}{minor}; sm_70+ required.\n"
    "Fix: Settings -> Accelerator -> 'GPU T4 x2', then re-run."
)
print("GPU OK.")

### Step 2: Locate both inputs

In [ ]:
import os
import sys


def find(predicate, label):
    hits = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        if predicate(root, files):
            hits.append(root)
    if not hits:
        raise AssertionError(f"{label} not attached. Searched /kaggle/input")
    return hits


assets = find(
    lambda _root, files: "testset.json" in files and "domain_eval.py" in files,
    "v3 eval assets",
)[0]
sys.path.insert(0, assets)
print("eval assets :", assets)

adapters = [
    root
    for root in find(
        lambda _root, files: "adapter_config.json" in files, "trained v3 adapter"
    )
    if "checkpoint" not in root
]
expected = [
    path
    for path in adapters
    if os.path.basename(path.rstrip("/")) == "Llama-3.1-8B-IOS-Risk-v1"
]
if len(expected) != 1:
    raise AssertionError(f"Expected exactly one v3 adapter; found {adapters}")
ADAPTER = expected[0]
print("adapter     :", ADAPTER)

### Step 3: Install (same pins as training)

In [ ]:
!pip install -q \
    "transformers==5.5.0" \
    "datasets==4.3.0" \
    "trl==0.24.0" \
    "bitsandbytes==0.50.1" \
    "xformers==0.0.34" \
    "peft>=0.18.0" \
    unsloth unsloth_zoo 2>&1 | tail -6
print("install done")

### Step 4: Score the BASE model

The control. Any score for the fine-tune is meaningless without it.

In [ ]:
# Importing FastLanguageModel first applies Unsloth's patches.
from unsloth import FastLanguageModel  # noqa: F401
from domain_eval import run

base_summary = run(
    model_id="unsloth/Meta-Llama-3.1-8B-Instruct",
    tag="base",
    testset_path=os.path.join(assets, "testset.json"),
    out_dir="/kaggle/working/eval_results",
)

### Step 5: Score the TUNED adapter

In [ ]:
tuned_summary = run(
    model_id=ADAPTER,
    tag="tuned",
    testset_path=os.path.join(assets, "testset.json"),
    out_dir="/kaggle/working/eval_results",
)

### Step 6: Verdict

In [ ]:
import json


def row(label, base, tuned, target=None, lower_is_better=False):
    passed = tuned <= target if lower_is_better else tuned > target
    flag = "" if target is None else ("   PASS" if passed else "   FAIL")
    print(
        f"{label:<23} base {base:>7.4f} tuned {tuned:>7.4f} delta {tuned - base:>+7.4f}{flag}"
    )


base_risk = base_summary["risk_assessment"]
tuned_risk = tuned_summary["risk_assessment"]
print("RISK ASSESSMENT (50 independently authored cases)")
row("tier accuracy", base_risk["tier_accuracy"], tuned_risk["tier_accuracy"], 0.70)
row("average quality", base_risk["avg_quality"], tuned_risk["avg_quality"], 0.60)
row("evidence rate", base_risk["evidence_rate"], tuned_risk["evidence_rate"])
row("action accuracy", base_risk["action_accuracy"], tuned_risk["action_accuracy"])
row(
    "unsupported claims",
    base_risk["unsupported_claim_rate"],
    tuned_risk["unsupported_claim_rate"],
    0.05,
    lower_is_better=True,
)

base_class = base_summary["classification"]
tuned_class = tuned_summary["classification"]
print("\nCLASSIFICATION (200 held-out source records)")
for metric in ("precision", "recall", "f1"):
    row(metric, base_class[metric], tuned_class[metric])
print(
    f"unparseable — base {base_class['unparseable']}, tuned {tuned_class['unparseable']}"
)

base_reg = base_summary["regulatory_recall"]
tuned_reg = tuned_summary["regulatory_recall"]
print("\nREGULATORY RECALL (26 entirely held-out CFR sections)")
row("citation accuracy", base_reg["citation_accuracy"], tuned_reg["citation_accuracy"])
print("\nPROJECT 03:", "PASS" if tuned_summary["passes_project03"] else "FAIL")

with open("/kaggle/working/eval_results/comparison.json", "w") as handle:
    json.dump({"base": base_summary, "tuned": tuned_summary}, handle, indent=2)

### Step 7: Read actual outputs

Metrics hide behaviour. The v2 smoke test scored correctly on all three probes
but emitted `Probability of fraud: 89.4%` and `Total score: 68` — numbers with no
basis in the training data or the input. Aggregates do not surface that.

In [ ]:
import json

with open("/kaggle/working/eval_results/eval_tuned.json") as handle:
    rows = json.load(handle)["risk_rows"]

for result in rows[:5]:
    mark = "OK" if result["quality_score"] == 1.0 else "REVIEW"
    print(f"[{mark}] expected {result['expected_tier']} got {result['predicted_tier']}")
    print("input:   ", result["input"])
    print("response:", result["response"][:400].replace("\n", " "))
    print("unsupported:", result["unsupported_claims"])
    print()

suspect = [result for result in rows if result["unsupported_claims"]]
print(f"responses with unsupported claims: {len(suspect)}/{len(rows)}")